# Common Libraries

In [1]:
import os, sys, shutil, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import opensim as osim

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module/")

from sj_array import reorient_array
from sj_datastructure import make_3d_dataset
from sj_visualization import plot_3D_dataset, plot_3D_datasets
from biomechanics_util import get_muscle_path, get_muscle_path_world

# Params

In [141]:
osim_file_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_fixed_41_normal/temp.osim"
muscle_names = ["DELT1"]
mujoco_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_fixed_41_normal/temp_cvt1.xml"

osim_coords = "IAL"
myoarm_coords = "RSP"
mujoco_coords = "IRA"

# Load opensim

In [142]:
# Load model and state
model = osim.Model(osim_file_path)
state = model.initSystem()

# Calculate muscle locations
osim_muscle_coords = {}
for muscle_name in muscle_names:
    osim_muscle_coords[muscle_name] = []
    muscle = model.getMuscles().get(muscle_name)
    geom_path = muscle.getGeometryPath()
    path_points = geom_path.getCurrentPath(state)
    
    for i in range(path_points.getSize()):
        pt = path_points.get(i)
        loc = pt.getLocationInGround(state)
        osim_muscle_coords[muscle_name].append([loc[0], loc[1], loc[2]])

# Preprocessing
for muscle_name in muscle_names:
    osim_muscle_coords[muscle_name] = np.array(osim_muscle_coords[muscle_name])
    osim_muscle_coords[muscle_name] = osim_muscle_coords[muscle_name][None, :, :]

# Make dataset for visualization
osim_muscle_ds_list = []
for muscle_name in muscle_names:
    coords = osim_muscle_coords[muscle_name]
    n_d, n_l, n_c = coords.shape
    osim_muscle_ds = make_3d_dataset(data = coords,
                                     wrapping_dataset_name = "3D",
                                     element_dataset_names = ["Times", "Labels", "Coords"],
                                     dataset1_dim_names = np.arange(n_d),
                                     dataset2_dim_names = np.arange(n_l).astype(str),
                                     dataset3_dim_names = ["X", "Y", "Z"])
    osim_muscle_ds_list.append(osim_muscle_ds)

osim_ds_names = [f"osim_{muscle_name}" for muscle_name in muscle_names]

[info] Loaded model Right from file /mnt/ext1/seojin/temp/MOBL_ARMS_fixed_41_normal/temp.osim


In [143]:
plot_3D_datasets(osim_muscle_ds_list, ds_coord_order = osim_coords, dataset_names = muscle_names)

# Mujoco

In [144]:
mujoco_muscle_ds_list = []
for muscle_name in muscle_names:
    df = get_muscle_path_world(mujoco_path, muscle_name)
    muscle_pos = df[["world_x", "world_y", "world_z"]].to_numpy()
    n_t, n_l, n_c = muscle_pos[None, :, :].shape
    muscle_ds = make_3d_dataset(data = muscle_pos[None, :, :],
                                wrapping_dataset_name = "3D",
                                element_dataset_names = ["Times", "Labels", "Coords"],
                                dataset1_dim_names = np.arange(n_t),
                                dataset2_dim_names = list(df["site_name"].to_numpy()),
                                dataset3_dim_names = ["X", "Y", "Z"])
    mujoco_muscle_ds_list.append(muscle_ds)

mujoco_ds_names = [f"mjc_{muscle_name}" for muscle_name in muscle_names]

In [145]:
plot_3D_datasets(mujoco_muscle_ds_list, mujoco_coords, dataset_names = muscle_names)

# Myoarm

In [146]:
myoarm_model_path = "/home/seojin/biomechanics_tools/myo_sim/arm/myoarm.xml"

myoarm_muscle_ds_list = []
for muscle_name in muscle_names:
    df = get_muscle_path_world(myoarm_model_path, muscle_name)
    muscle_pos = df[["world_x", "world_y", "world_z"]].to_numpy()
    n_t, n_l, n_c = muscle_pos[None, :, :].shape
    muscle_ds = make_3d_dataset(data = muscle_pos[None, :, :],
                                wrapping_dataset_name = "3D",
                                element_dataset_names = ["Times", "Labels", "Coords"],
                                dataset1_dim_names = np.arange(n_t),
                                dataset2_dim_names = list(df["site_name"].to_numpy()),
                                dataset3_dim_names = ["X", "Y", "Z"])
    myoarm_muscle_ds_list.append(muscle_ds)

myoarm_ds_names = [f"myo_{muscle_name}" for muscle_name in muscle_names]

In [147]:
plot_3D_datasets(myoarm_muscle_ds_list, myoarm_coords, dataset_names = muscle_names)

# Align

In [128]:
# offset
"""
myoarm_model = mujoco.MjModel.from_xml_path(myoarm_model_path)
myoarm_data = mujoco.MjData(myoarm_model)
myoarm_data.qpos[:] = 0
mujoco.mj_forward(myoarm_model, myoarm_data)

thorax_id = mujoco.mj_name2id(myoarm_model, mujoco.mjtObj.mjOBJ_BODY, "thorax")
myoarm_base_pos = myoarm_data.xpos[thorax_id]

mujoco_model = mujoco.MjModel.from_xml_path(mujoco_path)
mujoco_data = mujoco.MjData(mujoco_model)
mujoco_data.qpos[:] = 0
mujoco.mj_forward(mujoco_model, mujoco_data)
thorax_id = mujoco.mj_name2id(mujoco_model, mujoco.mjtObj.mjOBJ_BODY, "thorax")
mujoco_base_pos = mujoco_data.xpos[thorax_id]

offset = mujoco_base_pos - myoarm_base_pos
"""

'\nmyoarm_model = mujoco.MjModel.from_xml_path(myoarm_model_path)\nmyoarm_data = mujoco.MjData(myoarm_model)\nmyoarm_data.qpos[:] = 0\nmujoco.mj_forward(myoarm_model, myoarm_data)\n\nthorax_id = mujoco.mj_name2id(myoarm_model, mujoco.mjtObj.mjOBJ_BODY, "thorax")\nmyoarm_base_pos = myoarm_data.xpos[thorax_id]\n\nmujoco_model = mujoco.MjModel.from_xml_path(mujoco_path)\nmujoco_data = mujoco.MjData(mujoco_model)\nmujoco_data.qpos[:] = 0\nmujoco.mj_forward(mujoco_model, mujoco_data)\nthorax_id = mujoco.mj_name2id(mujoco_model, mujoco.mjtObj.mjOBJ_BODY, "thorax")\nmujoco_base_pos = mujoco_data.xpos[thorax_id]\n\noffset = mujoco_base_pos - myoarm_base_pos\n'

In [148]:
# Align coords
osim_muscle_ds_list_ = copy.deepcopy(osim_muscle_ds_list)
for position_ds in osim_muscle_ds_list_:
    position_ds["3D"].data = reorient_array(position_ds["3D"].data, osim_coords, mujoco_coords)

myoarm_muscle_ds_list_ = copy.deepcopy(myoarm_muscle_ds_list)
for position_ds in myoarm_muscle_ds_list_:
    position_ds["3D"].data = reorient_array(position_ds["3D"].data, myoarm_coords, mujoco_coords)

offset = mujoco_muscle_ds_list[0]["3D"].to_numpy()[0][0] - myoarm_muscle_ds_list_[0]["3D"].to_numpy()[0][0]
myoarm_muscle_ds_list_ = copy.deepcopy(myoarm_muscle_ds_list_)
for position_ds in myoarm_muscle_ds_list_:
    position_ds["3D"].data += offset

# Plot

In [149]:
plot_3D_datasets(mujoco_muscle_ds_list + myoarm_muscle_ds_list_,
                 ds_coord_order = mujoco_coords,
                 dataset_names = mujoco_ds_names + myoarm_ds_names)

In [150]:
plot_3D_datasets(osim_muscle_ds_list_ + mujoco_muscle_ds_list + myoarm_muscle_ds_list_,
                 ds_coord_order = mujoco_coords,
                 dataset_names = osim_ds_names + mujoco_ds_names + myoarm_ds_names)

In [140]:
osim_muscle_ds_list_[0]["3D"].to_numpy()

array([[[-0.02243322, -0.19558882, -0.0022259 ],
        [-0.06134502, -0.14756277, -0.00333538],
        [-0.06385115, -0.1439686 , -0.00467416],
        [-0.08182516, -0.11319011, -0.02776022]]])

In [22]:
len(osim_ds_names)

1